# Data cleaning Desafío 2

## Import libraries

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import date
from pathlib import Path

# Files

## Files path

In [ ]:

DATA_RRHH_RAW_FILE = '../../Data/raw/raw_data_01062026.csv'
DATA_RRHH_OLD_RAW_FILE = '../../Data/raw/raw_data_25052026.csv'
DATA_OUTPUT_FILE = '../../Data/clean/clean_data_01062026.csv'

def get_import_date(file_path):
    date_text = Path(file_path).stem.split('_')[-1]
    return pd.to_datetime(date_text, format='%d%m%Y').strftime('%Y-%m-%d')

IMPORTACION_ACTUAL = get_import_date(DATA_RRHH_RAW_FILE)
IMPORTACION_ANTERIOR = get_import_date(DATA_RRHH_OLD_RAW_FILE)

## Read files

In [5]:
df_RRHH = pd.read_csv(DATA_RRHH_RAW_FILE)
df_RRHH_old = pd.read_csv(DATA_RRHH_OLD_RAW_FILE)

# Procedencia de cada registro: si la fila exacta ya estaba en el CSV anterior,
# queda con fecha 2026-05-25; si aparece por primera vez esta semana, con 2026-06-01.
comparison_cols = df_RRHH.columns.intersection(df_RRHH_old.columns).tolist()
current_keys = df_RRHH[comparison_cols].apply(tuple, axis=1)
old_keys = df_RRHH_old[comparison_cols].apply(tuple, axis=1)

current_occurrence = current_keys.groupby(current_keys).cumcount() + 1
old_counts = old_keys.value_counts()

mask_existing = current_occurrence <= current_keys.map(old_counts).fillna(0).astype(int)
df_RRHH['importacion'] = np.where(mask_existing, IMPORTACION_ANTERIOR, IMPORTACION_ACTUAL)

print('Registros cargados:', df_RRHH.shape[0])
print('Columnas:', df_RRHH.shape[1])
display(df_RRHH['importacion'].value_counts().rename_axis('importacion').reset_index(name='registros'))

df_RRHH.sample(5)


Registros cargados: 845
Columnas: 22


,importacion,registros
0,2026-05-25,740
1,2026-06-01,105


,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
583,22,23,3,6,2,179,26,9,30,"222,196",...,3,0,0,0,0,56,171,19,2,2026-05-25
679,24,28,9,3,1,246,25,16,41,"261,756",...,1,0,1,0,0,67,170,23,1,2026-05-25
652,36,28,10,5,4,118,13,18,50,"265,017",...,1,1,1,0,0,98,178,31,1,2026-05-25
326,15,28,8,5,1,291,31,12,40,"249,797",...,1,1,1,0,1,73,171,25,4,2026-05-25
293,20,28,3,6,2,260,50,11,36,"343,253",...,1,4,1,0,0,65,168,23,4,2026-05-25


In [6]:
df_RRHH_old.sample(5)


,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
304,15,28,10,3,4,291,31,12,40,"265,017",...,0,1,1,1,0,1,73,171,25,4
336,5,26,7,4,1,235,20,13,43,"264,604",...,0,1,1,1,0,0,106,167,38,4
490,28,28,3,6,3,225,26,9,28,"343,253",...,0,1,1,0,0,2,69,169,24,2
3,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112
271,10,25,8,2,1,361,52,3,28,"205,917",...,0,1,1,1,0,4,80,172,27,7


# Data cleaning

In [5]:
df_RRHH.describe().T #Traspone columna

,count,mean,std,min,25%,50%,75%,max
ID,845.0,26.128994,26.374062,1.0,10.0,20.0,33.0,136.0
Reason_absence,845.0,19.280473,8.296957,0.0,13.0,23.0,26.0,28.0
Month_absence,845.0,6.315976,3.457020,0.0,3.0,6.0,9.0,12.0
Day_week,845.0,3.926627,1.417331,2.0,3.0,4.0,5.0,6.0
Seasons,845.0,2.540828,1.111040,1.0,2.0,3.0,4.0,4.0
Transportation_expense,845.0,220.482840,67.416501,118.0,179.0,225.0,260.0,388.0
Distance_Residence_Work,845.0,29.418935,14.740235,5.0,16.0,26.0,49.0,52.0
Service_time,845.0,12.578698,4.424460,1.0,9.0,12.0,16.0,29.0
Age,845.0,36.443787,6.555322,27.0,31.0,37.0,40.0,58.0
Hit_target,845.0,94.572781,3.798349,81.0,93.0,95.0,97.0,100.0


In [8]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   ID                       845 non-null    int64
 1   Reason_absence           845 non-null    int64
 2   Month_absence            845 non-null    int64
 3   Day_week                 845 non-null    int64
 4   Seasons                  845 non-null    int64
 5   Transportation_expense   845 non-null    int64
 6   Distance_Residence_Work  845 non-null    int64
 7   Service_time             845 non-null    int64
 8   Age                      845 non-null    int64
 9   Work_load_Average_day    845 non-null    str  
 10  Hit_target               845 non-null    int64
 11  Disciplinary_failure     845 non-null    int64
 12  Education                845 non-null    int64
 13  Son                      845 non-null    int64
 14  Social_drinker           845 non-null    int64
 15  Social_smoker    

## Check de Registros

In [7]:
def control_registros(df):

    fecha_actual = date.today()

    total_registros = df.shape[0]
    ID_unicos = df['ID'].nunique()
    ID_extra = total_registros - ID_unicos
    ID_duplicados_exactos = df[df.duplicated(keep=False)].drop_duplicates().shape[0]


    print('--------------------\nCONTROL DE REGISTROS\n--------------------')
    print(f'Fecha: {fecha_actual}\n--------------------')
    print(f'TOTAL REGISTROS: {total_registros}')
    print(f'ID ÚNICOS: {ID_unicos}')
    print(f'ID EXTRAS: {ID_extra}')
    print(f'ID DUPLICADOS EXACTOS: {ID_duplicados_exactos}')

In [8]:
control_registros(df_RRHH)

--------------------
CONTROL DE REGISTROS
--------------------
Fecha: 2026-06-03
--------------------
TOTAL REGISTROS: 845
ID ÚNICOS: 136
ID EXTRAS: 709
ID DUPLICADOS EXACTOS: 26


## Duplicated consistency

In [9]:
ID_distintos_con_duplicados = df_RRHH.loc[df_RRHH.duplicated(keep=False), 'ID'].nunique()
ID_distintos_con_duplicados_old = df_RRHH_old.loc[df_RRHH_old.duplicated(keep=False), 'ID'].nunique()
print(f'Esta semana los id con duplicados son: {ID_distintos_con_duplicados} y la semana pasada eran: {ID_distintos_con_duplicados_old}')

Esta semana los id con duplicados son: 9 y la semana pasada eran: 9


In [10]:
grupos_filas_duplicadas= df_RRHH[df_RRHH.duplicated(keep=False)].drop_duplicates().shape[0]
grupos_filas_duplicadas_old= df_RRHH_old[df_RRHH_old.duplicated(keep=False)].drop_duplicates().shape[0]
print(f'Esta semana los grupos de filas duplicadas son: {grupos_filas_duplicadas} y la semana pasada eran: {grupos_filas_duplicadas_old}')

Esta semana los grupos de filas duplicadas son: 26 y la semana pasada eran: 26


In [11]:
#Duplicados actuales y su conteo
df_RRHH[df_RRHH.duplicated(keep=False)].value_counts().reset_index(name='n_duplicados')

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion,n_duplicados
0,3,27,2,4,2,179,51,18,38,"251,818",...,0,1,0,0,89,170,31,3,2026-05-25,4
1,22,27,4,6,3,179,26,9,30,"246,288",...,0,0,0,0,56,171,19,2,2026-05-25,4
2,3,27,2,6,2,179,51,18,38,"251,818",...,0,1,0,0,89,170,31,3,2026-05-25,3
3,3,27,3,5,2,179,51,18,38,"222,196",...,0,1,0,0,89,170,31,3,2026-05-25,3
4,3,27,2,4,2,179,51,18,38,"264,249",...,0,1,0,0,89,170,31,2,2026-05-25,3
5,3,27,3,4,2,179,51,18,38,"222,196",...,0,1,0,0,89,170,31,2,2026-05-25,3
6,10,22,12,4,4,361,52,3,28,"261,306",...,1,1,0,4,80,172,27,8,2026-05-25,2
7,34,23,10,3,4,118,10,10,37,"253,465",...,0,0,0,0,83,172,28,3,2026-05-25,2
8,22,23,5,4,3,179,26,9,30,"246,074",...,0,0,0,0,56,171,19,3,2026-05-25,2
9,28,23,12,4,4,225,26,9,28,"280,549",...,1,0,0,2,69,169,24,3,2026-05-25,2


Comparison between old and new dataset

In [12]:
comparison = pd.DataFrame({
    "old": df_RRHH_old.loc[df_RRHH_old.duplicated(keep=False), 'ID'].value_counts(),
    "new": df_RRHH.loc[df_RRHH.duplicated(keep=False), 'ID'].value_counts()
}).fillna(0).astype(int)

comparison["diff"] = comparison["new"] - comparison["old"]

comparison.sort_values("diff", ascending=False)

,old,new,diff
ID,,,
3,26,26,0
34,10,10,0
22,8,8,0
28,4,4,0
24,4,4,0
10,2,2,0
27,2,2,0
5,2,2,0
15,2,2,0


In [13]:
comparison = pd.DataFrame({
    "old": df_RRHH_old.loc[df_RRHH_old.duplicated(keep=False), 'ID'].value_counts(),
    "new": df_RRHH.loc[df_RRHH.duplicated(keep=False), 'ID'].value_counts()
}).fillna(0).astype(int)

comparison["diff"] = comparison["new"] - comparison["old"]

comparison = comparison[comparison["diff"] != 0]
comparison.sort_values("diff", ascending=False)

,old,new,diff
ID,,,


- Se han introducido nuevos duplicados: +3 IDs nuevos con duplicados: 9, 14, 36
- Se han incrementado repeticiones en algunos IDs: Ej. ID 9 pasa de 0 → 4 duplicados
- La estructura de duplicados ha cambiado: 31 patrones distintos ahora vs 26 antes

In [14]:
df_RRHH[df_RRHH['ID'] == 28][df_RRHH[df_RRHH['ID'] == 28].duplicated(keep=False)]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
410,28,23,12,4,4,225,26,9,28,"280,549",...,1,1,0,0,2,69,169,24,3,2026-05-25
412,28,23,12,4,4,225,26,9,28,"280,549",...,1,1,0,0,2,69,169,24,3,2026-05-25
616,28,23,11,4,4,225,26,9,28,"306,345",...,1,1,0,0,2,69,169,24,1,2026-05-25
617,28,23,11,4,4,225,26,9,28,"306,345",...,1,1,0,0,2,69,169,24,1,2026-05-25


## Etiqueta de origen Input-day

df_RRHH_old["input_day"] = "2025-05-25"
df_RRHH["input_day"] = "2026-06-01"

In [15]:
df_RRHH.sample(5)

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
251,5,26,5,4,3,235,20,13,43,"237,656",...,1,1,1,0,0,106,167,38,8,2026-05-25
650,3,28,10,4,4,179,51,18,38,"265,017",...,1,0,1,0,0,89,170,31,1,2026-05-25
232,3,13,2,3,2,179,51,18,38,"264,249",...,1,0,1,0,0,89,170,31,8,2026-05-25
785,82,13,11,5,4,179,51,18,38,"306,345",...,1,0,1,0,0,89,170,31,8,2026-06-01
728,3,0,6,6,3,179,51,18,38,"253,957",...,1,0,1,0,0,89,170,31,0,2026-05-25


In [16]:
df_RRHH_old

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
0,14,11,11,2,4,155,12,14,34,"284,031",...,0,1,2,1,0,0,95,196,25,120
1,36,13,4,4,3,118,13,18,50,"239,409",...,0,1,1,1,0,0,98,178,31,120
2,9,6,7,3,1,228,14,16,58,"264,604",...,0,1,2,0,0,1,65,172,22,120
3,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112
4,9,12,3,3,2,228,14,16,58,"222,196",...,0,1,2,0,0,1,65,172,22,112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
735,2,0,6,2,3,235,29,12,48,"275,089",...,1,1,1,0,1,5,88,163,33,0
736,21,0,6,2,3,268,11,8,33,"275,089",...,1,2,0,0,0,0,79,178,25,0
737,4,0,0,3,1,118,14,13,40,"271,219",...,0,1,1,1,0,8,98,170,34,0
738,8,0,0,4,2,231,35,14,39,"271,219",...,0,1,2,1,0,2,100,170,35,0


df_merged = df_RRHH_old.merge(
    df_RRHH,
    how='left',
    indicator=True
)
df_merged

df_merged['date_status'] = df_merged['_merge'].map({
    'both': '25-05-2026',
    'left_only': '01062026'
})
df_merged

df_merged = df_merged.drop(columns='_merge')
df_merged

Concat datasets

df_RRHH_ = pd.concat([df_RRHH_old, df_RRHH], ignore_index=True)

Eliminate duplicates in the updated dataframe

df_RRHH_= df_RRHH.sort_values("input_day")

df_RRHH_ = df_RRHH.drop_duplicates(
    subset=df_RRHH_old.columns,  # sin input_day
    keep="last"
)

df_RRHH.info()

df_RRHH_["input_day"] = pd.to_datetime(df_RRHH["input_day"])

df_RRHH.sample(5)

df_RRHH_['input_day'].value_counts()

In [17]:
# Revisión de procedencia ya calculada al cargar los datos.
df_RRHH['importacion'].value_counts().rename_axis('importacion').reset_index(name='registros')


Registros ya presentes en el CSV anterior: 740
Registros nuevos del CSV actual: 105


In [18]:
df_RRHH.sample(5)


,ID,importacion,already_in_old
262,34,2026-05-25,True
348,34,2026-05-25,True
228,12,2026-05-25,True
255,15,2026-05-25,True
553,3,2026-05-25,True
453,3,2026-05-25,True
431,3,2026-05-25,True
276,28,2026-05-25,True
313,13,2026-05-25,True
127,24,2026-05-25,True


In [19]:
df_RRHH.shape


,importacion,registros
0,2026-05-25,740
1,2026-06-01,105


In [20]:
df_RRHH.columns


,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
427,33,25,3,3,2,248,25,14,47,"222,196",...,1,2,0,0,1,86,165,32,3,2026-05-25
628,28,23,2,4,2,225,26,9,28,"302,585",...,1,1,0,0,2,69,169,24,1,2026-05-25
533,22,23,9,2,1,179,26,9,30,"261,756",...,3,0,0,0,0,56,171,19,2,2026-05-25
421,3,27,2,5,2,179,51,18,38,"264,249",...,1,0,1,0,0,89,170,31,3,2026-05-25
226,13,26,11,6,4,369,17,12,31,"268,519",...,1,3,1,0,0,70,169,25,8,2026-05-25


Rows fully existing in old

In [21]:
mask_full_row_old = mask_existing


Rows where ID exists in old

In [22]:
mask_id_old = df_RRHH['ID'].isin(df_RRHH_old['ID'])



These are the problematic ones:

In [23]:
# IDs que ya existían en el CSV anterior, pero con registros nuevos o modificados en el CSV actual.
df_false_old = df_RRHH[mask_id_old & ~mask_full_row_old].copy()

df_false_old


,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
840,14,11,11,2,4,155,12,14,34,"284,031",...,1,2,1,0,0,95,196,25,120,2026-06-01
841,36,13,4,4,3,118,13,18,50,"239,409",...,1,1,1,0,0,98,178,31,120,2026-06-01
842,9,6,7,3,1,228,14,16,58,"264,604",...,1,2,0,0,1,65,172,22,120,2026-06-01
843,28,9,7,3,1,225,26,9,28,"230,290",...,1,1,0,0,2,69,169,24,112,2026-06-01
844,9,12,3,3,2,228,14,16,58,"222,196",...,1,2,0,0,1,65,172,22,112,2026-06-01


In [24]:
df_RRHH['importacion'].value_counts().rename_axis('importacion').reset_index(name='registros')


,importacion,registros
0,2026-05-25,740
1,2026-06-01,105


## Change Work_load_Average_day to float (, to .)

In [25]:
df_RRHH['Work_load_Average_day'] = df_RRHH['Work_load_Average_day'].str.replace(',', '.', regex=False).astype(float)

In [26]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       845 non-null    int64  
 1   Reason_absence           845 non-null    int64  
 2   Month_absence            845 non-null    int64  
 3   Day_week                 845 non-null    int64  
 4   Seasons                  845 non-null    int64  
 5   Transportation_expense   845 non-null    int64  
 6   Distance_Residence_Work  845 non-null    int64  
 7   Service_time             845 non-null    int64  
 8   Age                      845 non-null    int64  
 9   Work_load_Average_day    845 non-null    float64
 10  Hit_target               845 non-null    int64  
 11  Disciplinary_failure     845 non-null    int64  
 12  Education                845 non-null    int64  
 13  Son                      845 non-null    int64  
 14  Social_drinker           845 non-null

## Month_absence 0 - to nan

Check df

In [27]:
df_RRHH[df_RRHH['Month_absence'] == 0]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
737,4,0,0,3,1,118,14,13,40,271.219,...,1,1,1,0,8,98,170,34,0,2026-05-25
738,8,0,0,4,2,231,35,14,39,271.219,...,1,2,1,0,2,100,170,35,0,2026-05-25
739,35,0,0,6,3,179,45,14,53,271.219,...,1,1,0,0,1,77,175,25,0,2026-05-25


In [28]:
df_RRHH[(df_RRHH['Reason_absence'] == 0) & (df_RRHH['Disciplinary_failure'] == 0)]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
737,4,0,0,3,1,118,14,13,40,271.219,...,1,1,1,0,8,98,170,34,0,2026-05-25
738,8,0,0,4,2,231,35,14,39,271.219,...,1,2,1,0,2,100,170,35,0,2026-05-25
739,35,0,0,6,3,179,45,14,53,271.219,...,1,1,0,0,1,77,175,25,0,2026-05-25


Change

In [29]:
df_RRHH['Month_absence'] = df_RRHH['Month_absence'].replace(0, np.nan)

Check change correct

In [30]:
df_RRHH['Month_absence'].unique()

array([11.,  4.,  7.,  3.,  6., 12., 10.,  5.,  8.,  9.,  1.,  2., nan])

In [31]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       845 non-null    int64  
 1   Reason_absence           845 non-null    int64  
 2   Month_absence            842 non-null    float64
 3   Day_week                 845 non-null    int64  
 4   Seasons                  845 non-null    int64  
 5   Transportation_expense   845 non-null    int64  
 6   Distance_Residence_Work  845 non-null    int64  
 7   Service_time             845 non-null    int64  
 8   Age                      845 non-null    int64  
 9   Work_load_Average_day    845 non-null    float64
 10  Hit_target               845 non-null    int64  
 11  Disciplinary_failure     845 non-null    int64  
 12  Education                845 non-null    int64  
 13  Son                      845 non-null    int64  
 14  Social_drinker           845 non-null

## BMI - changed due to inconsistencies and add one decimal (medically relevant)

In [32]:
df_bmi = df_RRHH[['Weight','Height', 'Body_mass_index']].copy()
df_bmi

,Weight,Height,Body_mass_index
0,95,196,25
1,98,178,31
2,65,172,22
3,69,169,24
4,65,172,22
...,...,...,...
840,95,196,25
841,98,178,31
842,65,172,22
843,69,169,24


Since we have both weight and height, calculate BMI

In [33]:
df_bmi['BMI_calculated'] = (
    df_bmi['Weight'] / ((df_bmi['Height'] / 100) ** 2)
)

In [34]:
df_bmi['BMI_calculated_round'] = (
    df_bmi['Weight'] / ((df_bmi['Height'] / 100) ** 2)
).round().astype(int)

In [35]:
df_bmi['BMI_calculated_round_1'] = (
    df_bmi['Weight'] / ((df_bmi['Height'] / 100) ** 2)
).round(1)

In [36]:
df_bmi[df_bmi['BMI_calculated_round'] != df_bmi['Body_mass_index']] #filas donde las columnas no coinciden

,Weight,Height,Body_mass_index,BMI_calculated,BMI_calculated_round,BMI_calculated_round_1
50,88,172,29,29.745809,30,29.7
64,88,172,29,29.745809,30,29.7
89,88,172,29,29.745809,30,29.7
145,88,172,29,29.745809,30,29.7
178,88,172,29,29.745809,30,29.7
182,88,172,29,29.745809,30,29.7
193,88,172,29,29.745809,30,29.7
196,88,172,29,29.745809,30,29.7
218,75,178,25,23.671254,24,23.7
222,75,178,25,23.671254,24,23.7


Change (Insert new column calculated next to previous column)

In [37]:
df_RRHH.insert(20,'BMI_calculated',(
    df_RRHH['Weight'] / ((df_RRHH['Height'] / 100) ** 2)
).round(1))

Check change

In [38]:
df_RRHH[['ID','BMI_calculated','Body_mass_index']]

,ID,BMI_calculated,Body_mass_index
0,14,24.7,25
1,36,30.9,31
2,9,22.0,22
3,28,24.2,24
4,9,22.0,22
...,...,...,...
840,14,24.7,25
841,36,30.9,31
842,9,22.0,22
843,28,24.2,24


In [39]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       845 non-null    int64  
 1   Reason_absence           845 non-null    int64  
 2   Month_absence            842 non-null    float64
 3   Day_week                 845 non-null    int64  
 4   Seasons                  845 non-null    int64  
 5   Transportation_expense   845 non-null    int64  
 6   Distance_Residence_Work  845 non-null    int64  
 7   Service_time             845 non-null    int64  
 8   Age                      845 non-null    int64  
 9   Work_load_Average_day    845 non-null    float64
 10  Hit_target               845 non-null    int64  
 11  Disciplinary_failure     845 non-null    int64  
 12  Education                845 non-null    int64  
 13  Son                      845 non-null    int64  
 14  Social_drinker           845 non-null

## Delete row of ID 29 with different demographics

In [40]:
df_RRHH[df_RRHH['ID'] == 29]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
252,29,14,5.0,5,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,8,2026-05-25
253,29,22,5.0,6,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,8,2026-05-25
441,29,19,5.0,4,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,3,2026-05-25
555,29,28,2.0,6,2,225,15,15,41,264.249,...,2,1,0,2,94,182,28,28.4,2,2026-05-25
698,29,0,9.0,2,4,225,26,9,28,241.476,...,1,0,0,2,69,169,24,24.2,0,2026-05-25


In [41]:
df_RRHH[df_RRHH['ID'] == 1]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
50,1,13,6.0,6,3,235,11,14,37,377.550,...,1,0,0,1,88,172,29,29.7,16,2026-05-25
64,1,22,7.0,2,1,235,11,14,37,239.554,...,1,0,0,1,88,172,29,29.7,8,2026-05-25
89,1,26,12.0,4,4,235,11,14,37,261.306,...,1,0,0,1,88,172,29,29.7,8,2026-05-25
145,1,19,8.0,5,1,235,11,14,37,265.615,...,1,0,0,1,88,172,29,29.7,8,2026-05-25
178,1,22,3.0,2,2,235,11,14,37,244.387,...,1,0,0,1,88,172,29,29.7,8,2026-05-25
182,1,21,3.0,5,2,235,11,14,37,244.387,...,1,0,0,1,88,172,29,29.7,8,2026-05-25
193,1,1,5.0,2,3,235,11,14,37,246.074,...,1,0,0,1,88,172,29,29.7,8,2026-05-25
196,1,13,6.0,3,1,235,11,14,37,253.957,...,1,0,0,1,88,172,29,29.7,8,2026-05-25
247,1,22,4.0,6,3,235,11,14,37,246.288,...,1,0,0,1,88,172,29,29.7,8,2026-05-25
250,1,22,5.0,2,3,235,11,14,37,237.656,...,1,0,0,1,88,172,29,29.7,8,2026-05-25


In [42]:
df_RRHH[(df_RRHH['Age'] == 28) & (df_RRHH['Distance_Residence_Work'] == 26) & (df_RRHH['Service_time'] == 9)
        & (df_RRHH['Weight'] == 69) & (df_RRHH['Height'] == 169) & (df_RRHH['Body_mass_index'] == 24) & (df_RRHH['Son'] == 1)]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
3,28,9,7.0,3,1,225,26,9,28,230.290,...,1,0,0,2,69,169,24,24.2,112,2026-05-25
47,28,11,3.0,4,3,225,26,9,28,343.253,...,1,0,0,2,69,169,24,24.2,16,2026-05-25
115,28,11,3.0,2,3,225,26,9,28,343.253,...,1,0,0,2,69,169,24,24.2,8,2026-05-25
116,28,11,3.0,3,3,225,26,9,28,343.253,...,1,0,0,2,69,169,24,24.2,8,2026-05-25
123,28,19,5.0,3,3,225,26,9,28,378.884,...,1,0,0,2,69,169,24,24.2,8,2026-05-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
815,112,7,3.0,2,2,225,26,9,28,222.196,...,1,0,0,2,69,169,24,24.2,8,2026-06-01
820,117,28,3.0,2,3,225,26,9,28,343.253,...,1,0,0,2,69,169,24,24.2,1,2026-06-01
838,135,19,5.0,3,3,225,26,9,28,378.884,...,1,0,0,2,69,169,24,24.2,8,2026-06-01
839,136,23,12.0,4,4,225,26,9,28,280.549,...,1,0,0,2,69,169,24,24.2,3,2026-06-01


Decidimos eliminar el registro 698 como un error en el registro de los datos

In [43]:
df_RRHH = df_RRHH.drop(index=698)


Check

In [44]:
df_RRHH[df_RRHH['ID'] == 29]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
252,29,14,5.0,5,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,8,2026-05-25
253,29,22,5.0,6,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,8,2026-05-25
441,29,19,5.0,4,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,3,2026-05-25
555,29,28,2.0,6,2,225,15,15,41,264.249,...,2,1,0,2,94,182,28,28.4,2,2026-05-25


## New entries updated this week

In [45]:
df_RRHH['ID'].max()

np.int64(136)

In [46]:
df_RRHH['ID'].unique()

array([ 14,  36,   9,  28,  11,  13,  34,  22,  26,  20,  10,  15,  17,
        24,   3,  18,   7,   1,  30,   5,   6,   2,  31,  27,  33,  23,
        21,  25,  12,  32,  16,  29,  19,   8,   4,  35,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136])

In [47]:
workers_old = 36
workers_updated = df_RRHH['ID'].nunique()

new_added= workers_updated - workers_old
new_added

100

## Reason absence 0 to nan - It is not a code and mainly corresponding to disciplinary failure 1

In [48]:
df_RRHH[df_RRHH['Reason_absence'] == 0]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
696,36,0,7.0,3,1,118,13,18,50,239.554,...,1,1,0,0,98,178,31,30.9,0,2026-05-25
697,20,0,9.0,2,4,260,50,11,36,241.476,...,4,1,0,0,65,168,23,23.0,0,2026-05-25
699,11,0,9.0,3,4,289,36,13,33,241.476,...,2,1,0,1,90,172,30,30.4,0,2026-05-25
700,36,0,9.0,3,4,118,13,18,50,241.476,...,1,1,0,0,98,178,31,30.9,0,2026-05-25
701,13,0,9.0,4,4,369,17,12,31,241.476,...,3,1,0,0,70,169,25,24.5,0,2026-05-25
702,36,0,10.0,4,4,118,13,18,50,253.465,...,1,1,0,0,98,178,31,30.9,0,2026-05-25
704,2,0,4.0,2,3,235,29,12,48,326.452,...,1,0,1,5,88,163,33,33.1,0,2026-05-25
705,7,0,5.0,4,3,279,5,14,39,378.884,...,2,1,1,0,68,168,24,24.1,0,2026-05-25
706,18,0,5.0,4,3,330,16,4,28,378.884,...,0,0,0,0,84,182,25,25.4,0,2026-05-25
707,23,0,5.0,4,3,378,49,11,36,378.884,...,2,0,1,4,65,174,21,21.5,0,2026-05-25


Change

In [49]:
df_RRHH['Reason_absence'] = df_RRHH['Reason_absence'].replace(0, np.nan)

Check change correct

In [50]:
df_RRHH['Reason_absence'].unique()

array([11., 13.,  6.,  9., 12., 19., 18.,  1., 10., 14.,  7., 28.,  2.,
       26., 23., 22., 21., 24., 17.,  8.,  5., 15.,  4., 25.,  3., 27.,
       16., nan])

In [51]:
df_RRHH.info()

<class 'pandas.DataFrame'>
Index: 844 entries, 0 to 844
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       844 non-null    int64  
 1   Reason_absence           802 non-null    float64
 2   Month_absence            841 non-null    float64
 3   Day_week                 844 non-null    int64  
 4   Seasons                  844 non-null    int64  
 5   Transportation_expense   844 non-null    int64  
 6   Distance_Residence_Work  844 non-null    int64  
 7   Service_time             844 non-null    int64  
 8   Age                      844 non-null    int64  
 9   Work_load_Average_day    844 non-null    float64
 10  Hit_target               844 non-null    int64  
 11  Disciplinary_failure     844 non-null    int64  
 12  Education                844 non-null    int64  
 13  Son                      844 non-null    int64  
 14  Social_drinker           844 non-null    i

## Seasons to correct season by month

Seasons (summer (1), autumn (2), winter (3), spring (4))  
España:  
- 1 (winter): 6 - 9 (originally 1 - not change) 
- 2 (spring): 9 - 12 (originally 4 - CHANGE) 
- 3 (summer): 12 - 3 (originally 2 - CHANGE)  
- 4 (autumn): 3 - 6 (originally 3 - CHANGE) 

Brasil se encuentra en el hemisferio sur y sus cuatro estaciones siguen el calendario austral:
- Primavera: del 22 de septiembre al 20 de diciembre.
- Verano: del 21 de diciembre al 19 de marzo (temporada alta y de lluvias en muchas zonas). 
- Otoño: del 20 de marzo al 20 de junio.
- Invierno: del 21 de junio al 21 de septiembre.

In [52]:
df_RRHH.groupby('Seasons')['Month_absence'].unique()

Seasons
1     [7.0, 8.0, 9.0, 6.0, nan]
2    [3.0, 12.0, 1.0, 2.0, nan]
3     [4.0, 3.0, 6.0, 5.0, nan]
4       [11.0, 12.0, 10.0, 9.0]
Name: Month_absence, dtype: object

Change

In [53]:
#Ordenamos los meses correctamente a Estacionalidad Brasil
df_RRHH['Seasons'] = df_RRHH['Seasons'].replace({
    4.0: 2.0,
    2.0: 3.0,
    3.0: 4.0
})

Check

In [54]:
df_RRHH.groupby('Seasons')['Month_absence'].unique()

Seasons
1     [7.0, 8.0, 9.0, 6.0, nan]
2       [11.0, 12.0, 10.0, 9.0]
3    [3.0, 12.0, 1.0, 2.0, nan]
4     [4.0, 3.0, 6.0, 5.0, nan]
Name: Month_absence, dtype: object

### Cambiar los meses para que las estaciones Brasileñas coincidad con las Españolas

Desplazamos 6 meses, es un cambio de hemisferio norte al sur

- Del 6 - 9 (Invierno Brasil) cambio a 12 - 3 (Invierno España)
- Del 9 - 12 (Primavera Brasil) cambio a 3 - 6 (Primavera España)
- Del 12 - 3 (Verano Brasil) cambio a 6 - 9 (Verano España)
- Del 3 - 6 (Otoño Brasil) cambio a 9 - 12 (Otoño españa)

In [55]:
mask = df_RRHH['Month_absence'].notna() #Cromprovacion de que el mas existe

df_RRHH.loc[mask, 'Month_absence'] = (
    (df_RRHH.loc[mask, 'Month_absence'] + 5) % 12
) + 1

In [56]:
df_RRHH.groupby('Seasons')['Month_absence'].unique()

Seasons
1      [1.0, 2.0, 3.0, 12.0, nan]
2            [5.0, 6.0, 4.0, 3.0]
3       [9.0, 6.0, 7.0, 8.0, nan]
4    [10.0, 9.0, 12.0, 11.0, nan]
Name: Month_absence, dtype: object

Se realizó un desplazamiento de seis meses en la variable Month_absence con el fin de adaptar la temporalidad del dataset brasileño al contexto español. De este modo, los patrones estacionales observados en variables como Hit_target o el absentismo conservan su interpretación climática y temporal, permitiendo que fenómenos asociados al verano, invierno o periodos vacacionales se mantengan coherentes en la simulación del contexto español.

## Create a column with the labels corresponding to the numbers

Dictionaries

In [57]:
reason_labels = {
    1: "Enfermedades infecciosas y parasitarias",
    2: "Neoplasias",
    3: "Sangre e inmunidad",
    4: "Endocrinas, nutricionales y metabólicas",
    5: "Trastornos mentales y del comportamiento",
    6: "Sistema nervioso",
    7: "Ojo y anexos",
    8: "Oído y apófisis mastoides",
    9: "Sistema circulatorio",
    10: "Sistema respiratorio",
    11: "Sistema digestivo",
    12: "Piel y tejido subcutáneo",
    13: "Sistema musculoesquelético",
    14: "Sistema genitourinario",
    15: "Embarazo, parto y puerperio",
    16: "Afecciones perinatales",
    17: "Malformaciones congénitas",
    18: "Síntomas y hallazgos no clasificados",
    19: "Lesiones, intoxicaciones y consecuencias externas",
    20: "Causas externas de morbilidad y mortalidad",
    21: "Factores de salud y contacto sanitario",
    22: "Seguimiento de paciente",
    23: "Consulta médica",
    24: "Donación de sangre",
    25: "Examen de laboratorio",
    26: "Ausencia injustificada",
    27: "Fisioterapia",
    28: "Consulta dental",
}

month_labels = {
    1.0: "Enero", 2.0: "Febrero", 3.0: "Marzo", 4.0: "Abril", 5.0: "Mayo", 6.0: "Junio",
    7.0: "Julio", 8.0: "Agosto", 9.0: "Septiembre", 10.0: "Octubre", 11.0: "Noviembre", 12.0: "Diciembre",
}

day_labels = {2: "Lunes", 3: "Martes", 4: "Miércoles", 5: "Jueves", 6: "Viernes"}
season_labels = {1: "Invierno", 2: "Primavera", 3: "Verano", 4: "Otoño"}
education_labels = {1: "Secundaria", 2: "Grado", 3: "Posgrado", 4: "Máster/Doctorado"}

Create function to do it for each column

In [58]:
def insert_column_labels(df, column: str, name_new_column: str, labels: dict):
    
    col_index = df.columns.get_loc(column)

    df.insert(
        col_index + 1,
        name_new_column,
        df[column].replace(labels)
    )

In [59]:
df_RRHH.columns

Index(['ID', 'Reason_absence', 'Month_absence', 'Day_week', 'Seasons',
       'Transportation_expense', 'Distance_Residence_Work', 'Service_time',
       'Age', 'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure',
       'Education', 'Son', 'Social_drinker', 'Social_smoker', 'Pet', 'Weight',
       'Height', 'Body_mass_index', 'BMI_calculated', 'Absenteeism_hours',
       'importacion'],
      dtype='str')

Change

In [60]:
insert_column_labels(df_RRHH,'Month_absence','Month_absence_name',month_labels)
insert_column_labels(df_RRHH,'Reason_absence','Reason_absence_name',reason_labels)
insert_column_labels(df_RRHH,'Day_week','Day_week_name',day_labels)
insert_column_labels(df_RRHH,'Seasons','Seasons_name',season_labels)
insert_column_labels(df_RRHH,'Education','Education_name',education_labels)

In [61]:
df_RRHH.columns

Index(['ID', 'Reason_absence', 'Reason_absence_name', 'Month_absence',
       'Month_absence_name', 'Day_week', 'Day_week_name', 'Seasons',
       'Seasons_name', 'Transportation_expense', 'Distance_Residence_Work',
       'Service_time', 'Age', 'Work_load_Average_day', 'Hit_target',
       'Disciplinary_failure', 'Education', 'Education_name', 'Son',
       'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height',
       'Body_mass_index', 'BMI_calculated', 'Absenteeism_hours',
       'importacion'],
      dtype='str')

In [62]:
df_RRHH.info()

<class 'pandas.DataFrame'>
Index: 844 entries, 0 to 844
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       844 non-null    int64  
 1   Reason_absence           802 non-null    float64
 2   Reason_absence_name      802 non-null    object 
 3   Month_absence            841 non-null    float64
 4   Month_absence_name       841 non-null    object 
 5   Day_week                 844 non-null    int64  
 6   Day_week_name            844 non-null    object 
 7   Seasons                  844 non-null    int64  
 8   Seasons_name             844 non-null    object 
 9   Transportation_expense   844 non-null    int64  
 10  Distance_Residence_Work  844 non-null    int64  
 11  Service_time             844 non-null    int64  
 12  Age                      844 non-null    int64  
 13  Work_load_Average_day    844 non-null    float64
 14  Hit_target               844 non-null    i

In [63]:
df_RRHH.sample(5)

,ID,Reason_absence,Reason_absence_name,Month_absence,Month_absence_name,Day_week,Day_week_name,Seasons,Seasons_name,Transportation_expense,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
792,89,22.0,Seguimiento de paciente,6.0,Junio,4,Miércoles,2,Primavera,361,...,1,1,0,4,80,172,27,27.0,8,2026-06-01
433,33,23.0,Consulta médica,9.0,Septiembre,5,Jueves,3,Verano,248,...,2,0,0,1,86,165,32,31.6,3,2026-05-25
821,118,27.0,Fisioterapia,9.0,Septiembre,5,Jueves,3,Verano,179,...,0,1,0,0,89,170,31,30.8,3,2026-06-01
324,24,28.0,Consulta dental,2.0,Febrero,3,Martes,1,Invierno,246,...,0,1,0,0,67,170,23,23.2,4,2026-05-25
527,24,28.0,Consulta dental,1.0,Enero,3,Martes,1,Invierno,246,...,0,1,0,0,67,170,23,23.2,2,2026-05-25


Se han añadido registros duplicados de los IDs que ya teniamos antes del input semanal

## Outliers

In [64]:
def detectar_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return df[(df[col] < lower) | (df[col] > upper)]

In [65]:
continuous_cols = [
    'Age',
    'Height',
    'Weight',
    'Body_mass_index',
    'Absenteeism_hours',
    'Transportation_expense',
    'Distance_Residence_Work',
    'Service_time',
    'Work_load_Average_day',
    'Hit_target'
]

In [66]:
resumen_outliers = []

for col in continuous_cols:
    resumen_outliers.append({
        'Variable': col,
        'Recuento_outliers': detectar_outliers_iqr(df_RRHH, col).shape[0],
        'Min': df_RRHH[col].min(),
        'Max': df_RRHH[col].max(),
        'Media': df_RRHH[col].mean().round(2),
        'Mediana': df_RRHH[col].median()
    })

resumen_outliers = (
    pd.DataFrame(resumen_outliers)
    .sort_values('Recuento_outliers', ascending=False)
)

resumen_outliers

,Variable,Recuento_outliers,Min,Max,Media,Mediana
1,Height,134,163.000,196.000,172.11,170.500
4,Absenteeism_hours,51,0.000,120.000,7.38,3.000
8,Work_load_Average_day,37,205.917,378.884,271.96,264.604
9,Hit_target,23,81.000,100.000,94.58,95.000
0,Age,10,27.000,58.000,36.45,37.000
7,Service_time,7,1.000,29.000,12.58,12.500
5,Transportation_expense,3,118.000,388.000,220.48,225.000
2,Weight,0,56.000,108.000,79.03,83.000
3,Body_mass_index,0,19.000,38.000,26.68,25.000
6,Distance_Residence_Work,0,5.000,52.000,29.42,26.000


# Save to csv file

In [67]:
# df_RRHH.to_csv(DATA_OUTPUT_FILE,encoding='utf-8', index=False)